In [1]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt các thư viện AI CHƯA CÓ sẵn
!pip install transformers accelerate

Mounted at /content/drive


In [2]:
import os

DATA_DIR = "/content/drive/MyDrive/Studies/Data_Retrieval/Data_extractor/DataMovie"
BATCH_SIZE = 16

# Kiểm tra thư mục có tồn tại hay không
kiem_tra = os.path.exists(DATA_DIR)
print(f"Thư mục tồn tại: {kiem_tra}")

Thư mục tồn tại: True


In [3]:
import torch
from PIL import Image, ImageEnhance
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import glob
import os
import re
from tqdm import tqdm

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TỚI GOOGLE DRIVE
# ==============================================================================
DATA_DIR = "/content/drive/MyDrive/Studies/Data_Retrieval/Data_extractor/DataMovie"
BATCH_SIZE = 16

# ==============================================================================
# 2. KHỞI TẠO MÔ HÌNH BLIP-2 VÀO GPU (Đã bỏ device_map='auto' chống lỗi)
# ==============================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang chạy trên thiết bị: {device.upper()}")
print("[*] Đang nạp BLIP-2 Model...")

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
).to(device)
print("[+] Nạp mô hình thành công!\n")

# ==============================================================================
# 3. HÀM LỌC VÀ LÀM SẠCH KẾT QUẢ VĂN BẢN
# ==============================================================================
def clean_and_validate_caption(raw_text):
    text = raw_text.strip()

    # Cắt bỏ nếu AI bị lặp lại cụm mồi "The scene shows"
    if text.lower().startswith("the scene shows"):
        text = text[15:].strip()

    # Loại bỏ các tiền tố lửng lơ vô nghĩa
    text = re.sub(r"^(is|shows|depicts|showing)\s+", "", text, flags=re.IGNORECASE)

    # Loại bỏ rác ký tự lặp
    text = re.sub(r"[_\.\-=\*|]{2,}", "", text).strip()

    # Lọc ngôn ngữ lạ (Chỉ chấp nhận ký tự chuẩn ASCII Tiếng Anh)
    if not all(32 <= ord(char) <= 126 for char in text):
        return ""

    # Loại bỏ các câu quá ngắn (vì Prompt ép tả chi tiết, nếu ngắn là lỗi)
    if len(text) < 10 or text.lower() in ["none", "nothing", "the scene"]:
        return ""

    # Gắn lại cụm từ mồi để tạo thành một đoạn văn hoàn chỉnh
    # Ví dụ: "The scene shows a man in a red jacket standing..."
    if text:
        text = "The scene shows " + text[0].lower() + text[1:]

    return text

# ==============================================================================
# 4. QUÉT THƯ MỤC VÀ XỬ LÝ THEO LÔ (BATCH PROCESSING)
# ==============================================================================
if not os.path.exists(DATA_DIR):
    print(f"[-] LỖI: Không tìm thấy thư mục {DATA_DIR}.")
else:
    movie_folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]

    for movie_name in movie_folders:
        picture_dir = os.path.join(DATA_DIR, movie_name, "picture")

        if not os.path.exists(picture_dir):
            continue

        image_paths = glob.glob(os.path.join(picture_dir, "*.jpg"))
        if not image_paths:
            continue

        output_file = os.path.join(DATA_DIR, movie_name, f"{movie_name}_captions.txt")
        print(f"\n==================================================")
        print(f"[*] Đang xử lý GHI ĐÈ phim: {movie_name} ({len(image_paths)} ảnh)")

        with open(output_file, "w", encoding="utf-8") as f:
            for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Tiến độ"):
                batch_paths = image_paths[i : i + BATCH_SIZE]

                images = []
                valid_paths = []
                for p in batch_paths:
                    try:
                        img = Image.open(p).convert('RGB')

                        # Tăng sáng & tương phản trị các phim tối/CGI
                        enhancer = ImageEnhance.Brightness(img)
                        img_bright = enhancer.enhance(1.3)
                        enhancer_contrast = ImageEnhance.Contrast(img_bright)
                        img_final = enhancer_contrast.enhance(1.1)

                        images.append(img_final)
                        valid_paths.append(p)
                    except Exception:
                        pass

                if not images: continue

                # SỬ DỤNG PROMPT DENSE: Ép bóc tách Hành động, Đồ vật, Màu sắc, Bối cảnh
                prompts = [
                    "Question: Provide a highly detailed visual description of this scene. You must explicitly describe the main physical actions, specific objects, exact colors of clothing, the background environment, and any visible text or logos. Answer: The scene shows"
                ] * len(images)

                inputs = processor(images=images, text=prompts, return_tensors="pt").to(device, torch.float16)

                # BỘ SIÊU THAM SỐ ĐÃ ĐƯỢC CĂN CHỈNH
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=65,       # Đủ dài để tả chi tiết 4 trường thông tin
                    min_new_tokens=20,       # Ép buộc phải nói nhiều, cấm tả qua loa
                    repetition_penalty=1.15  # Xóa bỏ tình trạng lặp từ
                )
                generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

                for path, text in zip(valid_paths, generated_texts):
                    file_name = os.path.basename(path)

                    clean_text = clean_and_validate_caption(text)

                    if clean_text:
                        f.write(f"{file_name} | {clean_text}\n")
                    else:
                        # Dự phòng an toàn cho SBERT khi khung hình quá nhiễu/tối
                        f.write(f"{file_name} | The scene shows a dark or fast-moving physical action.\n")

    print("\n[+] HOÀN THÀNH GHI ĐÈ TOÀN BỘ QUÁ TRÌNH TRÍCH XUẤT ĐA THUỘC TÍNH!")

[*] Đang chạy trên thiết bị: CUDA
[*] Đang nạp BLIP-2 Model...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...


[*] Đang xử lý GHI ĐÈ phim: Star_Wars_Episode_VIII_The_Last_Jedi (51 ảnh)


Tiến độ: 100%|██████████| 4/4 [00:26<00:00,  6.59s/it]



[*] Đang xử lý GHI ĐÈ phim: Cocoon_Aru_Natsu_no_Shoujo_tachi_yori (31 ảnh)


Tiến độ: 100%|██████████| 2/2 [00:16<00:00,  8.12s/it]


[+] HOÀN THÀNH GHI ĐÈ TOÀN BỘ QUÁ TRÌNH TRÍCH XUẤT ĐA THUỘC TÍNH!
